# SafeFundus — Notebook 08: MC Dropout Uncertainty

## Purpose

The current SafeFundus classifier is frozen as:

> **Swin-Tiny + Asymmetric Loss + explicit NORMAL output**

and Notebook 07B established the primary open-set energy component:

> **Normal-Aware Dual-Mode JointEnergy**

This notebook adds the second independent safety signal:

# MC Dropout Uncertainty

No classifier retraining occurs here.

---

## MC Dropout setup

The trained classifier head contains:

\[
\text{Dropout}(p=0.30) \rightarrow \text{Linear}(768,15)
\]

During normal inference, dropout is disabled.

For uncertainty estimation, we keep the backbone in evaluation mode but reactivate the trained prediction-head dropout and perform:

\[
M=20
\]

stochastic forward passes.

For class \(c\):

\[
\bar p_c(x)
=
\frac{1}{M}
\sum_{m=1}^{M}
p_c^{(m)}(x)
\]

and

\[
V_c(x)
=
\frac{1}{M}
\sum_{m=1}^{M}
\left(
p_c^{(m)}(x)-\bar p_c(x)
\right)^2
\]

The primary uncertainty score is:

\[
U_{MC}(x)
=
\frac{1}{15}
\sum_{c=1}^{15}V_c(x)
\]

using the NORMAL output plus all 14 known diseases.

We also save:

- disease-only variance;
- NORMAL variance;
- predictive entropy;

as diagnostics.

---

## Validation-only normalization

For later fusion, uncertainty is standardized using only familiar validation cases:

\[
U_N(x)
=
\frac{U_{MC}(x)-\mu_U}
{\sigma_U}
\]

where \(\mu_U,\sigma_U\) are estimated from the familiar validation set.

---

## Primary MC Dropout referral threshold

The threshold is selected using familiar validation cases only:

\[
\tau_U
=
Q_{0.90}(U_N^{val})
\]

This targets approximately a **10% validation false-referral rate**.

We also report 5% and 15% operating points as sensitivity analysis.

No unknown test image is used to tune:

- model weights;
- uncertainty normalization;
- uncertainty threshold;
- disease thresholds.

---

## Safety evaluation

For unknown-only cases we preserve the deterministic 06C disease decision and ask:

> Would MC Dropout uncertainty refer the case before the system could falsely reassure it as normal?

This isolates the safety contribution of the uncertainty module.

After Notebook 08, the next stage is:

\[
\text{Normal-Aware JointEnergy}
+
\text{MC Dropout uncertainty}
\rightarrow
\text{SafeFundus fusion}
\]


In [1]:
# ============================================================
# 1. Imports
# ============================================================

from pathlib import Path
import os
import json
import random
import shutil
import zipfile
import warnings
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import Swin_T_Weights
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


# 2. Required Kaggle Inputs

Attach:

1. **06C Swin-Tiny ASL Normal-Aware output**
2. **RFMiD dataset**

The notebook will automatically locate the 06C ZIP/output and the known RFMiD dataset path.


In [2]:
# ============================================================
# 2. Paths / configuration
# ============================================================

SOURCE_NAME = (
    "06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier"
)

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

DATA_ROOT = Path(
    "/kaggle/input/datasets/ozlemhakdagli/"
    "retinal-fundus-multi-disease-image-dataset-rfmid"
)

VAL_IMAGE_DIR = DATA_ROOT / "Validation_set"
TEST_IMAGE_DIR = DATA_ROOT / "Test_set"

NOTEBOOK_STEM = "08_MC_Dropout_Uncertainty"

OUTPUT_DIR = WORK_ROOT / NOTEBOOK_STEM
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
PER_SEED_DIR = OUTPUT_DIR / "per_seed"

for p in [
    OUTPUT_DIR,
    TABLE_DIR,
    FIG_DIR,
    PER_SEED_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

CONFIG = {
    "seeds": [42, 123, 2026],

    "mc_passes": 20,

    "image_size": 224,
    "batch_size": 32,
    "num_workers": 2,

    "dropout_probability": 0.30,

    "primary_val_false_referral_rate":
        0.10,

    "sensitivity_val_false_referral_rates": [
        0.05,
        0.10,
        0.15,
    ],

    "normal_class":
        "NORMAL",

    "known_diseases": [
        "DR", "MH", "ODC", "TSLN",
        "DN", "MYA", "ARMD", "BRVO",
        "ODP", "ODE", "LS", "RS",
        "CSR", "CRS",
    ],
}

NORMAL_CLASS = CONFIG[
    "normal_class"
]

KNOWN_DISEASES = CONFIG[
    "known_diseases"
]

MODEL_CLASSES = [
    NORMAL_CLASS,
    *KNOWN_DISEASES,
]

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "RFMiD dataset was not found at the expected path. "
        "Attach the RFMiD dataset used in the earlier notebooks."
    )

print(json.dumps(CONFIG, indent=2))
print("\nDEVICE:", DEVICE)


{
  "seeds": [
    42,
    123,
    2026
  ],
  "mc_passes": 20,
  "image_size": 224,
  "batch_size": 32,
  "num_workers": 2,
  "dropout_probability": 0.3,
  "primary_val_false_referral_rate": 0.1,
  "sensitivity_val_false_referral_rates": [
    0.05,
    0.1,
    0.15
  ],
  "normal_class": "NORMAL",
  "known_diseases": [
    "DR",
    "MH",
    "ODC",
    "TSLN",
    "DN",
    "MYA",
    "ARMD",
    "BRVO",
    "ODP",
    "ODE",
    "LS",
    "RS",
    "CSR",
    "CRS"
  ]
}

DEVICE: cuda


# 3. Locate and Extract Notebook 06C Output


In [3]:
# ============================================================
# 3. Locate / extract 06C
# ============================================================

source_root = None

zip_candidates = list(
    INPUT_ROOT.rglob(
        f"{SOURCE_NAME}.zip"
    )
)

if zip_candidates:
    source_zip = zip_candidates[0]

    extract_dir = (
        WORK_ROOT /
        "notebook06c_for_mc_dropout"
    )

    if extract_dir.exists():
        shutil.rmtree(
            extract_dir
        )

    extract_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        source_zip,
        "r",
    ) as zf:
        zf.extractall(
            extract_dir
        )

    source_root = extract_dir

    print("Found 06C ZIP:")
    print(source_zip)

else:
    candidates = []

    for p in INPUT_ROOT.rglob(
        "validation_logits_predictions_15class.csv"
    ):
        if (
            "seed_42" in str(p)
            and "predictions" in str(p)
        ):
            candidates.append(
                p.parents[2]
            )

    if candidates:
        source_root = candidates[0]

        print(
            "Found extracted 06C output:"
        )
        print(source_root)

if source_root is None:
    raise FileNotFoundError(
        "Could not find 06C output. Attach "
        "06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier "
        "as a Kaggle input."
    )

print("\nUsing 06C source root:")
print(source_root)


Found 06C ZIP:
/kaggle/input/notebooks/shouravdebnath/06c-swintiny-asl-normalaware-multiseed-classifier/06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier.zip

Using 06C source root:
/kaggle/working/notebook06c_for_mc_dropout


# 4. Load 06C Metadata, Deterministic Predictions, and Thresholds

The saved deterministic 06C predictions are retained for the primary safety endpoint.

MC Dropout only determines **referral** here; it does not silently redefine the original closed-set disease decision.


In [4]:
# ============================================================
# 4. Load 06C seed data
# ============================================================

def load_seed_artifacts(seed):

    seed_root = (
        source_root /
        f"seed_{seed}"
    )

    pred_root = (
        seed_root /
        "predictions"
    )

    table_root = (
        seed_root /
        "tables"
    )

    checkpoint_path = (
        source_root /
        "checkpoints" /
        f"swin_tiny_asl_normalaware_seed_{seed}.pth"
    )

    paths = {
        "validation":
            pred_root /
            "validation_logits_predictions_15class.csv",

        "familiar":
            pred_root /
            "familiar_test_logits_predictions_15class.csv",

        "unknown":
            pred_root /
            "unknown_test_logits_predictions_15class.csv",

        "mixed":
            pred_root /
            "mixed_test_logits_predictions_15class.csv",

        "intermediate":
            pred_root /
            "intermediate_test_logits_predictions_15class.csv",

        "thresholds":
            table_root /
            "validation_thresholds_15class.csv",
    }

    required = [
        *paths.values(),
        checkpoint_path,
    ]

    missing = [
        str(p)
        for p in required
        if not p.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Missing required 06C files:\n"
            + "\n".join(missing)
        )

    result = {
        key: pd.read_csv(path)
        for key, path in paths.items()
    }

    result[
        "checkpoint_path"
    ] = checkpoint_path

    threshold_map = dict(
        zip(
            result["thresholds"]["Class"],
            result["thresholds"]["Threshold"],
        )
    )

    result[
        "threshold_map"
    ] = threshold_map

    return result


for seed in CONFIG["seeds"]:
    data = load_seed_artifacts(
        seed
    )

    print(
        f"Seed {seed}: "
        f"val={len(data['validation'])}, "
        f"familiar={len(data['familiar'])}, "
        f"unknown={len(data['unknown'])}, "
        f"mixed={len(data['mixed'])}, "
        f"intermediate={len(data['intermediate'])}"
    )


Seed 42: val=560, familiar=571, unknown=32, mixed=24, intermediate=13
Seed 123: val=560, familiar=571, unknown=32, mixed=24, intermediate=13
Seed 2026: val=560, familiar=571, unknown=32, mixed=24, intermediate=13


# 5. Build Image Tables Directly from the Saved 06C IDs

This guarantees that MC Dropout evaluates exactly the same cases used in the previous experiment.


In [5]:
# ============================================================
# 5. Image-table helper
# ============================================================

def normalize_id(x):
    return str(
        int(float(x))
    )


def attach_image_paths(
    dataframe,
    image_folder,
):
    out = dataframe[
        ["ID", "Group"]
    ].copy()

    out["ID"] = out[
        "ID"
    ].apply(
        normalize_id
    )

    out["image_path"] = (
        out["ID"]
        .apply(
            lambda x:
                str(
                    image_folder /
                    f"{x}.png"
                )
        )
    )

    missing = [
        p
        for p in out[
            "image_path"
        ]
        if not Path(p).exists()
    ]

    if missing:
        raise FileNotFoundError(
            f"{len(missing)} images missing. "
            f"Example: {missing[:3]}"
        )

    return out


# 6. Evaluation Transform and Dataset

The exact deterministic 224×224 ImageNet-normalized evaluation transform from Notebook 06C is reused.


In [6]:
# ============================================================
# 6. Transform / dataset
# ============================================================

weights = (
    Swin_T_Weights.IMAGENET1K_V1
)

imagenet_mean = (
    weights.transforms().mean
)

imagenet_std = (
    weights.transforms().std
)

eval_transform = transforms.Compose([
    transforms.Resize(
        (
            CONFIG["image_size"],
            CONFIG["image_size"],
        )
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std,
    ),
])


class FundusInferenceDataset(
    Dataset
):
    def __init__(
        self,
        dataframe,
        transform,
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
        )
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        image = self.transform(
            image
        )

        return {
            "image":
                image,
            "id":
                row["ID"],
            "group":
                row["Group"],
        }


def make_loader(
    dataframe,
):
    dataset = FundusInferenceDataset(
        dataframe,
        eval_transform,
    )

    return DataLoader(
        dataset,
        batch_size=
            CONFIG["batch_size"],
        shuffle=False,
        num_workers=
            CONFIG["num_workers"],
        pin_memory=True,
        persistent_workers=(
            CONFIG[
                "num_workers"
            ] > 0
        ),
    )


# 7. Reconstruct the Frozen 15-Output Swin-Tiny

No weights are changed.

The checkpoint architecture is:

\[
\text{Swin-Tiny}
\rightarrow
Dropout(0.30)
\rightarrow
15\text{ logits}
\]


In [7]:
# ============================================================
# 7. Model
# ============================================================

def build_swin_tiny():
    # weights=None because the trained checkpoint
    # completely restores the model parameters.
    model = models.swin_t(
        weights=None
    )

    in_features = (
        model.head.in_features
    )

    model.head = nn.Sequential(
        nn.Dropout(
            p=CONFIG[
                "dropout_probability"
            ]
        ),
        nn.Linear(
            in_features,
            len(MODEL_CLASSES),
        ),
    )

    return model


def load_model(
    checkpoint_path,
):
    model = build_swin_tiny()

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model = model.to(
        DEVICE
    )

    model.eval()

    return model


# 8. Enable Only the Trained Prediction-Head Dropout

We intentionally keep the Swin-Tiny feature extractor in evaluation mode.

Only the explicit trained head dropout is reactivated.

This avoids turning stochastic-depth training behavior back on during inference.


In [8]:
# ============================================================
# 8. Dropout control
# ============================================================

def enable_head_mc_dropout(
    model,
):
    model.eval()

    dropout_modules = []

    for module in model.head.modules():
        if isinstance(
            module,
            nn.Dropout,
        ):
            module.train()
            dropout_modules.append(
                module
            )

    if len(dropout_modules) == 0:
        raise RuntimeError(
            "No Dropout module found in model.head."
        )

    return len(
        dropout_modules
    )


# 9. MC Dropout Inference

For each image we save:

- predictive mean probability for all 15 classes;
- predictive variance for all 15 classes;
- mean variance across 15 classes;
- mean variance across 14 disease classes;
- NORMAL variance;
- predictive entropy.

The primary score remains:

\[
U_{MC}
=
\text{mean class variance over all 15 outputs}
\]


In [9]:
# ============================================================
# 9. MC inference
# ============================================================

def binary_entropy(
    p,
    eps=1e-8,
):
    p = np.clip(
        p,
        eps,
        1.0 - eps,
    )

    return (
        -p * np.log(p)
        -(1.0-p)
        * np.log(1.0-p)
    )


@torch.no_grad()
def mc_dropout_inference(
    model,
    loader,
    mc_passes,
):
    n_dropout = (
        enable_head_mc_dropout(
            model
        )
    )

    print(
        f"Active head Dropout modules: "
        f"{n_dropout}"
    )

    all_ids = []
    all_groups = []

    all_mean_prob = []
    all_var_prob = []

    for batch in tqdm(
        loader,
        desc=f"MC Dropout x{mc_passes}",
    ):
        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        stochastic_probs = []

        for _ in range(
            mc_passes
        ):
            with torch.cuda.amp.autocast(
                enabled=(
                    DEVICE.type
                    == "cuda"
                )
            ):
                logits = model(
                    images
                )

            probs = torch.sigmoid(
                logits
            )

            stochastic_probs.append(
                probs
                .float()
                .cpu()
                .numpy()
            )

        stacked = np.stack(
            stochastic_probs,
            axis=0,
        )

        mean_prob = (
            stacked.mean(axis=0)
        )

        var_prob = (
            stacked.var(
                axis=0,
                ddof=0,
            )
        )

        all_mean_prob.append(
            mean_prob
        )

        all_var_prob.append(
            var_prob
        )

        all_ids.extend(
            batch["id"]
        )

        all_groups.extend(
            batch["group"]
        )

    mean_prob = np.concatenate(
        all_mean_prob,
        axis=0,
    )

    var_prob = np.concatenate(
        all_var_prob,
        axis=0,
    )

    uncertainty_all15 = (
        var_prob.mean(
            axis=1
        )
    )

    uncertainty_disease14 = (
        var_prob[:, 1:]
        .mean(axis=1)
    )

    uncertainty_normal = (
        var_prob[:, 0]
    )

    predictive_entropy = (
        binary_entropy(
            mean_prob
        )
        .mean(axis=1)
    )

    return {
        "ids":
            all_ids,
        "groups":
            all_groups,
        "mean_prob":
            mean_prob,
        "var_prob":
            var_prob,
        "uncertainty_all15":
            uncertainty_all15,
        "uncertainty_disease14":
            uncertainty_disease14,
        "uncertainty_normal":
            uncertainty_normal,
        "predictive_entropy":
            predictive_entropy,
    }


# 10. OOD Metrics

Unknown-only images are the positive class.

Higher uncertainty means more likely unknown.


In [10]:
# ============================================================
# 10. OOD metrics
# ============================================================

def fpr_at_95_tpr(
    y_true,
    scores,
):
    fpr, tpr, thresholds = (
        roc_curve(
            y_true,
            scores,
        )
    )

    valid = np.where(
        tpr >= 0.95
    )[0]

    if len(valid) == 0:
        return np.nan

    return float(
        np.min(
            fpr[valid]
        )
    )


def ood_metrics(
    familiar_scores,
    unknown_scores,
):
    y_true = np.concatenate([
        np.zeros(
            len(familiar_scores),
            dtype=int,
        ),
        np.ones(
            len(unknown_scores),
            dtype=int,
        ),
    ])

    scores = np.concatenate([
        familiar_scores,
        unknown_scores,
    ])

    return {
        "unknown_auroc":
            float(
                roc_auc_score(
                    y_true,
                    scores,
                )
            ),
        "unknown_aupr":
            float(
                average_precision_score(
                    y_true,
                    scores,
                )
            ),
        "fpr_at_95_tpr":
            fpr_at_95_tpr(
                y_true,
                scores,
            ),
    }


# 11. Deterministic Closed-Set Safety Helper

The original 06C deterministic probabilities and validation disease thresholds are used to identify:

- no known disease predicted → closed-set rare→normal;
- known disease predicted → unknown forced into known.

The MC Dropout module then determines whether either outcome is intercepted by referral.


In [11]:
# ============================================================
# 11. Closed-set decisions
# ============================================================

def deterministic_unknown_decisions(
    unknown_df,
    threshold_map,
):
    disease_prob = np.stack(
        [
            unknown_df[
                f"prob_{c}"
            ].to_numpy(
                dtype=float
            )
            for c in KNOWN_DISEASES
        ],
        axis=1,
    )

    disease_thresholds = np.asarray(
        [
            threshold_map[c]
            for c in KNOWN_DISEASES
        ],
        dtype=float,
    )

    disease_pred = (
        disease_prob
        >= disease_thresholds[
            None, :
        ]
    )

    any_disease = (
        disease_pred.any(
            axis=1
        )
    )

    no_disease = (
        ~any_disease
    )

    normal_prob = (
        unknown_df[
            f"prob_{NORMAL_CLASS}"
        ].to_numpy(
            dtype=float
        )
    )

    normal_threshold = float(
        threshold_map[
            NORMAL_CLASS
        ]
    )

    explicit_normal = (
        normal_prob
        >= normal_threshold
    )

    return {
        "any_disease":
            any_disease,
        "no_disease":
            no_disease,
        "explicit_normal":
            explicit_normal,
    }


# 12. Export MC Results


In [12]:
# ============================================================
# 12. Export helper
# ============================================================

def mc_result_dataframe(
    result,
):
    df = pd.DataFrame({
        "ID":
            result["ids"],
        "Group":
            result["groups"],
        "U_MC_All15":
            result[
                "uncertainty_all15"
            ],
        "U_MC_Disease14":
            result[
                "uncertainty_disease14"
            ],
        "U_MC_Normal":
            result[
                "uncertainty_normal"
            ],
        "Predictive_Entropy":
            result[
                "predictive_entropy"
            ],
    })

    for c, class_name in enumerate(
        MODEL_CLASSES
    ):
        df[
            f"mc_mean_prob_{class_name}"
        ] = (
            result[
                "mean_prob"
            ][:, c]
        )

        df[
            f"mc_var_prob_{class_name}"
        ] = (
            result[
                "var_prob"
            ][:, c]
        )

    return df


# 13. Run MC Dropout Across All Three Seeds

Primary score:

\[
U_{MC}^{15}
\]

Primary operating point:

> 10% familiar validation false-referral target.

We additionally measure mixed and intermediate referral rates as exploratory results.


In [13]:
# ============================================================
# 13. Multi-seed MC Dropout experiment
# ============================================================

seed_summaries = []
threshold_sensitivity_rows = []

for seed in CONFIG["seeds"]:

    print("\n" + "#" * 80)
    print(
        f"MC DROPOUT — SEED {seed}"
    )
    print("#" * 80)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    artifacts = (
        load_seed_artifacts(
            seed
        )
    )

    # --------------------------------------------------------
    # Build exact image tables from saved 06C IDs.
    # --------------------------------------------------------

    val_images = attach_image_paths(
        artifacts[
            "validation"
        ],
        VAL_IMAGE_DIR,
    )

    familiar_images = (
        attach_image_paths(
            artifacts[
                "familiar"
            ],
            TEST_IMAGE_DIR,
        )
    )

    unknown_images = (
        attach_image_paths(
            artifacts[
                "unknown"
            ],
            TEST_IMAGE_DIR,
        )
    )

    mixed_images = (
        attach_image_paths(
            artifacts[
                "mixed"
            ],
            TEST_IMAGE_DIR,
        )
    )

    intermediate_images = (
        attach_image_paths(
            artifacts[
                "intermediate"
            ],
            TEST_IMAGE_DIR,
        )
    )

    loaders = {
        "validation":
            make_loader(
                val_images
            ),
        "familiar":
            make_loader(
                familiar_images
            ),
        "unknown":
            make_loader(
                unknown_images
            ),
        "mixed":
            make_loader(
                mixed_images
            ),
        "intermediate":
            make_loader(
                intermediate_images
            ),
    }

    model = load_model(
        artifacts[
            "checkpoint_path"
        ]
    )

    # --------------------------------------------------------
    # MC inference.
    # --------------------------------------------------------

    mc = {}

    for split_name in [
        "validation",
        "familiar",
        "unknown",
        "mixed",
        "intermediate",
    ]:
        print(
            f"\nSeed {seed} | "
            f"{split_name}"
        )

        mc[split_name] = (
            mc_dropout_inference(
                model,
                loaders[
                    split_name
                ],
                CONFIG[
                    "mc_passes"
                ],
            )
        )

    # --------------------------------------------------------
    # Validation-only uncertainty normalization.
    # --------------------------------------------------------

    val_raw = mc[
        "validation"
    ][
        "uncertainty_all15"
    ]

    mu_u = float(
        val_raw.mean()
    )

    sigma_u = float(
        val_raw.std(
            ddof=0
        )
    )

    if sigma_u < 1e-12:
        raise ValueError(
            "Validation MC uncertainty "
            "standard deviation is zero."
        )

    def normalize_u(x):
        return (
            x - mu_u
        ) / sigma_u

    val_u = normalize_u(
        mc["validation"][
            "uncertainty_all15"
        ]
    )

    familiar_u = normalize_u(
        mc["familiar"][
            "uncertainty_all15"
        ]
    )

    unknown_u = normalize_u(
        mc["unknown"][
            "uncertainty_all15"
        ]
    )

    mixed_u = normalize_u(
        mc["mixed"][
            "uncertainty_all15"
        ]
    )

    intermediate_u = normalize_u(
        mc["intermediate"][
            "uncertainty_all15"
        ]
    )

    # --------------------------------------------------------
    # Primary validation threshold.
    # --------------------------------------------------------

    target_frr = CONFIG[
        "primary_val_false_referral_rate"
    ]

    referral_threshold = float(
        np.quantile(
            val_u,
            1.0 - target_frr,
        )
    )

    val_ref = (
        val_u
        > referral_threshold
    )

    familiar_ref = (
        familiar_u
        > referral_threshold
    )

    unknown_ref = (
        unknown_u
        > referral_threshold
    )

    mixed_ref = (
        mixed_u
        > referral_threshold
    )

    intermediate_ref = (
        intermediate_u
        > referral_threshold
    )

    # --------------------------------------------------------
    # Familiar subgroups.
    # --------------------------------------------------------

    familiar_groups = np.asarray(
        mc["familiar"][
            "groups"
        ]
    )

    normal_mask = (
        familiar_groups
        == "Normal"
    )

    known_mask = (
        familiar_groups
        == "Known_Only"
    )

    # --------------------------------------------------------
    # Closed-set unknown outcomes from deterministic 06C.
    # --------------------------------------------------------

    decisions = (
        deterministic_unknown_decisions(
            artifacts[
                "unknown"
            ],
            artifacts[
                "threshold_map"
            ],
        )
    )

    residual = (
        ~unknown_ref
    )

    rare_to_normal = (
        residual
        & decisions[
            "no_disease"
        ]
    )

    explicit_normal_false_reassurance = (
        residual
        & decisions[
            "no_disease"
        ]
        & decisions[
            "explicit_normal"
        ]
    )

    forced_known = (
        residual
        & decisions[
            "any_disease"
        ]
    )

    # --------------------------------------------------------
    # OOD metrics.
    # --------------------------------------------------------

    primary_ood = (
        ood_metrics(
            familiar_u,
            unknown_u,
        )
    )

    # Diagnostic predictive-entropy discrimination.
    entropy_ood = (
        ood_metrics(
            mc["familiar"][
                "predictive_entropy"
            ],
            mc["unknown"][
                "predictive_entropy"
            ],
        )
    )

    # --------------------------------------------------------
    # Save per-image outputs.
    # --------------------------------------------------------

    seed_out = (
        PER_SEED_DIR /
        f"seed_{seed}"
    )

    seed_out.mkdir(
        parents=True,
        exist_ok=True,
    )

    for split_name in [
        "validation",
        "familiar",
        "unknown",
        "mixed",
        "intermediate",
    ]:
        export_df = (
            mc_result_dataframe(
                mc[
                    split_name
                ]
            )
        )

        split_u = {
            "validation":
                val_u,
            "familiar":
                familiar_u,
            "unknown":
                unknown_u,
            "mixed":
                mixed_u,
            "intermediate":
                intermediate_u,
        }[
            split_name
        ]

        export_df[
            "U_MC_Normalized"
        ] = split_u

        export_df[
            "Referral_Primary"
        ] = (
            split_u
            > referral_threshold
        ).astype(int)

        if split_name == "unknown":
            export_df[
                "ClosedSet_No_Disease"
            ] = (
                decisions[
                    "no_disease"
                ].astype(int)
            )

            export_df[
                "Explicit_Normal_Predicted"
            ] = (
                decisions[
                    "explicit_normal"
                ].astype(int)
            )

            export_df[
                "Rare_to_Normal_After_MC"
            ] = (
                rare_to_normal
                .astype(int)
            )

            export_df[
                "Forced_Known_After_MC"
            ] = (
                forced_known
                .astype(int)
            )

        export_df.to_csv(
            seed_out /
            f"{split_name}_mc_dropout_scores.csv",
            index=False,
        )

    # --------------------------------------------------------
    # Threshold sensitivity.
    # --------------------------------------------------------

    for requested_frr in CONFIG[
        "sensitivity_val_false_referral_rates"
    ]:
        threshold = float(
            np.quantile(
                val_u,
                1.0 - requested_frr,
            )
        )

        fam_ref_s = (
            familiar_u
            > threshold
        )

        unk_ref_s = (
            unknown_u
            > threshold
        )

        residual_s = (
            ~unk_ref_s
        )

        rare_s = (
            residual_s
            & decisions[
                "no_disease"
            ]
        )

        strict_s = (
            residual_s
            & decisions[
                "no_disease"
            ]
            & decisions[
                "explicit_normal"
            ]
        )

        threshold_sensitivity_rows.append({
            "seed":
                seed,

            "requested_val_false_referral_rate":
                requested_frr,

            "threshold_U_N":
                threshold,

            "actual_val_false_referral_rate":
                float(
                    (
                        val_u
                        > threshold
                    ).mean()
                ),

            "familiar_test_false_referral_rate":
                float(
                    fam_ref_s.mean()
                ),

            "normal_test_false_referral_rate":
                float(
                    fam_ref_s[
                        normal_mask
                    ].mean()
                ),

            "known_only_test_false_referral_rate":
                float(
                    fam_ref_s[
                        known_mask
                    ].mean()
                ),

            "unknown_referral_sensitivity":
                float(
                    unk_ref_s.mean()
                ),

            "post_referral_rare_to_normal_rate":
                float(
                    rare_s.mean()
                ),

            "post_referral_explicit_normal_false_reassurance":
                float(
                    strict_s.mean()
                ),
        })

    # --------------------------------------------------------
    # Seed summary.
    # --------------------------------------------------------

    seed_summary = {
        "seed":
            seed,

        "mc_passes":
            CONFIG[
                "mc_passes"
            ],

        "validation_uncertainty_mean":
            mu_u,

        "validation_uncertainty_sd":
            sigma_u,

        "primary_referral_threshold_U_N":
            referral_threshold,

        "actual_val_false_referral_rate":
            float(
                val_ref.mean()
            ),

        "familiar_test_false_referral_rate":
            float(
                familiar_ref.mean()
            ),

        "normal_test_false_referral_rate":
            float(
                familiar_ref[
                    normal_mask
                ].mean()
            ),

        "known_only_test_false_referral_rate":
            float(
                familiar_ref[
                    known_mask
                ].mean()
            ),

        "unknown_referral_sensitivity":
            float(
                unknown_ref.mean()
            ),

        "mixed_referral_rate":
            float(
                mixed_ref.mean()
            ),

        "intermediate_referral_rate":
            float(
                intermediate_ref.mean()
            ),

        "closedset_rare_to_normal_rate":
            float(
                decisions[
                    "no_disease"
                ].mean()
            ),

        "post_referral_rare_to_normal_rate":
            float(
                rare_to_normal.mean()
            ),

        "post_referral_explicit_normal_false_reassurance":
            float(
                explicit_normal_false_reassurance.mean()
            ),

        "post_referral_forced_known_rate":
            float(
                forced_known.mean()
            ),

        "unknown_auroc":
            primary_ood[
                "unknown_auroc"
            ],

        "unknown_aupr":
            primary_ood[
                "unknown_aupr"
            ],

        "fpr_at_95_tpr":
            primary_ood[
                "fpr_at_95_tpr"
            ],

        "entropy_unknown_auroc":
            entropy_ood[
                "unknown_auroc"
            ],

        "entropy_unknown_aupr":
            entropy_ood[
                "unknown_aupr"
            ],
    }

    seed_summaries.append(
        seed_summary
    )

    with open(
        seed_out /
        "mc_dropout_seed_summary.json",
        "w",
    ) as f:
        json.dump(
            seed_summary,
            f,
            indent=2,
        )

    # --------------------------------------------------------
    # Distribution figure.
    # --------------------------------------------------------

    plt.figure(
        figsize=(8, 5)
    )

    plt.hist(
        familiar_u,
        bins=30,
        density=True,
        alpha=0.6,
        label="Familiar test",
    )

    plt.hist(
        unknown_u,
        bins=15,
        density=True,
        alpha=0.6,
        label="Unknown-only test",
    )

    plt.axvline(
        referral_threshold,
        linestyle="--",
        label="Validation threshold",
    )

    plt.xlabel(
        "Normalized MC Dropout Uncertainty (U_N)"
    )

    plt.ylabel(
        "Density"
    )

    plt.title(
        f"MC Dropout Uncertainty — Seed {seed}"
    )

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIG_DIR /
        f"mc_dropout_distribution_seed_{seed}.png",
        dpi=200,
        bbox_inches="tight",
    )

    plt.close()

    print("\nSeed summary:")
    print(
        json.dumps(
            seed_summary,
            indent=2,
        )
    )

    del model
    del loaders
    del mc

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll three MC Dropout seeds completed.")



################################################################################
MC DROPOUT — SEED 42
################################################################################

Seed 42 | validation
Active head Dropout modules: 1


MC Dropout x20:   0%|          | 0/18 [00:00<?, ?it/s]


Seed 42 | familiar
Active head Dropout modules: 1


MC Dropout x20:   0%|          | 0/18 [00:00<?, ?it/s]


Seed 42 | unknown
Active head Dropout modules: 1


MC Dropout x20:   0%|          | 0/1 [00:00<?, ?it/s]

... (21 outputs left out)

MC Dropout x20:   0%|          | 0/1 [00:00<?, ?it/s]


Seed 2026 | intermediate
Active head Dropout modules: 1


MC Dropout x20:   0%|          | 0/1 [00:00<?, ?it/s]


Seed summary:
{
  "seed": 2026,
  "mc_passes": 20,
  "validation_uncertainty_mean": 0.0011506900191307068,
  "validation_uncertainty_sd": 0.0004246910393703729,
  "primary_referral_threshold_U_N": 1.3358689546585083,
  "actual_val_false_referral_rate": 0.1,
  "familiar_test_false_referral_rate": 0.07880910683012259,
  "normal_test_false_referral_rate": 0.0,
  "known_only_test_false_referral_rate": 0.10297482837528604,
  "unknown_referral_sensitivity": 0.375,
  "mixed_referral_rate": 0.2916666666666667,
  "intermediate_referral_rate": 0.46153846153846156,
  "closedset_rare_to_normal_rate": 0.3125,
  "post_referral_rare_to_normal_rate": 0.21875,
  "post_referral_explicit_normal_false_reassurance": 0.1875,
  "post_referral_forced_known_rate": 0.40625,
  "unknown_auroc": 0.7043016637478108,
  "unknown_aupr": 0.2027274719398226,
  "fpr_at_95_tpr": 0.8178633975481612,
  "entropy_unknown_auroc": 0.6335376532399298,
  "entropy_unknown_aupr": 0.07843713023198942
}

All three MC Dropout seeds c

# 14. Aggregate Multi-Seed MC Dropout Results


In [14]:
# ============================================================
# 14. Aggregate results
# ============================================================

seed_results_df = pd.DataFrame(
    seed_summaries
)

seed_results_df.to_csv(
    TABLE_DIR /
    "mc_dropout_all_seed_results.csv",
    index=False,
)

display(
    seed_results_df
)

AGGREGATE_METRICS = [
    "familiar_test_false_referral_rate",
    "normal_test_false_referral_rate",
    "known_only_test_false_referral_rate",
    "unknown_referral_sensitivity",
    "mixed_referral_rate",
    "intermediate_referral_rate",
    "closedset_rare_to_normal_rate",
    "post_referral_rare_to_normal_rate",
    "post_referral_explicit_normal_false_reassurance",
    "post_referral_forced_known_rate",
    "unknown_auroc",
    "unknown_aupr",
    "fpr_at_95_tpr",
    "entropy_unknown_auroc",
    "entropy_unknown_aupr",
]

aggregate_rows = []

for metric in AGGREGATE_METRICS:
    values = pd.to_numeric(
        seed_results_df[
            metric
        ],
        errors="coerce",
    )

    aggregate_rows.append({
        "Metric":
            metric,
        "Mean":
            float(
                values.mean()
            ),
        "SD":
            float(
                values.std(
                    ddof=1
                )
            ),
        "Min":
            float(
                values.min()
            ),
        "Max":
            float(
                values.max()
            ),
    })

aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_df.to_csv(
    TABLE_DIR /
    "mc_dropout_mean_sd.csv",
    index=False,
)

display(
    aggregate_df
)

print("\n" + "=" * 80)
print(
    "MC DROPOUT MULTI-SEED SUMMARY"
)
print("=" * 80)

for _, row in (
    aggregate_df.iterrows()
):
    print(
        f"{row['Metric']:55s}: "
        f"{row['Mean']:.6f} "
        f"± {row['SD']:.6f}"
    )


,seed,mc_passes,validation_uncertainty_mean,validation_uncertainty_sd,primary_referral_threshold_U_N,actual_val_false_referral_rate,familiar_test_false_referral_rate,normal_test_false_referral_rate,known_only_test_false_referral_rate,unknown_referral_sensitivity,...,intermediate_referral_rate,closedset_rare_to_normal_rate,post_referral_rare_to_normal_rate,post_referral_explicit_normal_false_reassurance,post_referral_forced_known_rate,unknown_auroc,unknown_aupr,fpr_at_95_tpr,entropy_unknown_auroc,entropy_unknown_aupr
0,42,20,0.001185,0.000341,1.301160,0.1,0.108581,0.0,0.141876,0.18750,...,0.384615,0.21875,0.21875,0.09375,0.59375,0.728930,0.135668,0.819615,0.595611,0.114941
1,123,20,0.001062,0.000477,1.315754,0.1,0.064799,0.0,0.084668,0.21875,...,0.153846,0.28125,0.12500,0.12500,0.65625,0.661066,0.128034,0.947461,0.595665,0.075830
2,2026,20,0.001151,0.000425,1.335869,0.1,0.078809,0.0,0.102975,0.37500,...,0.461538,0.31250,0.21875,0.18750,0.40625,0.704302,0.202727,0.817863,0.633538,0.078437


,Metric,Mean,SD,Min,Max
0,familiar_test_false_referral_rate,0.084063,0.022359,0.064799,0.108581
1,normal_test_false_referral_rate,0.000000,0.000000,0.000000,0.000000
2,known_only_test_false_referral_rate,0.109840,0.029215,0.084668,0.141876
3,unknown_referral_sensitivity,0.260417,0.100455,0.187500,0.375000
4,mixed_referral_rate,0.263889,0.086736,0.166667,0.333333
5,intermediate_referral_rate,0.333333,0.160128,0.153846,0.461538
6,closedset_rare_to_normal_rate,0.270833,0.047735,0.218750,0.312500
7,post_referral_rare_to_normal_rate,0.187500,0.054127,0.125000,0.218750
8,post_referral_explicit_normal_false_reassurance,0.135417,0.047735,0.093750,0.187500
9,post_referral_forced_known_rate,0.552083,0.130104,0.406250,0.656250



MC DROPOUT MULTI-SEED SUMMARY
familiar_test_false_referral_rate                      : 0.084063 ± 0.022359
normal_test_false_referral_rate                        : 0.000000 ± 0.000000
known_only_test_false_referral_rate                    : 0.109840 ± 0.029215
unknown_referral_sensitivity                           : 0.260417 ± 0.100455
mixed_referral_rate                                    : 0.263889 ± 0.086736
intermediate_referral_rate                             : 0.333333 ± 0.160128
closedset_rare_to_normal_rate                          : 0.270833 ± 0.047735
post_referral_rare_to_normal_rate                      : 0.187500 ± 0.054127
post_referral_explicit_normal_false_reassurance        : 0.135417 ± 0.047735
post_referral_forced_known_rate                        : 0.552083 ± 0.130104
unknown_auroc                                          : 0.698099 ± 0.034354
unknown_aupr                                           : 0.155476 ± 0.041098
fpr_at_95_tpr                                

# 15. Threshold Sensitivity Analysis

The 10% validation false-referral setting remains the primary point.

The 5% and 15% settings show the uncertainty-only referral trade-off.


In [15]:
# ============================================================
# 15. Sensitivity analysis
# ============================================================

sensitivity_df = pd.DataFrame(
    threshold_sensitivity_rows
)

sensitivity_df.to_csv(
    TABLE_DIR /
    "mc_dropout_threshold_sensitivity_by_seed.csv",
    index=False,
)

display(
    sensitivity_df
)

sensitivity_aggregate = (
    sensitivity_df
    .groupby(
        "requested_val_false_referral_rate"
    )
    .agg(
        familiar_false_referral_mean=(
            "familiar_test_false_referral_rate",
            "mean",
        ),
        familiar_false_referral_sd=(
            "familiar_test_false_referral_rate",
            "std",
        ),
        normal_false_referral_mean=(
            "normal_test_false_referral_rate",
            "mean",
        ),
        known_false_referral_mean=(
            "known_only_test_false_referral_rate",
            "mean",
        ),
        unknown_referral_sensitivity_mean=(
            "unknown_referral_sensitivity",
            "mean",
        ),
        unknown_referral_sensitivity_sd=(
            "unknown_referral_sensitivity",
            "std",
        ),
        rare_to_normal_mean=(
            "post_referral_rare_to_normal_rate",
            "mean",
        ),
        strict_false_reassurance_mean=(
            "post_referral_explicit_normal_false_reassurance",
            "mean",
        ),
    )
    .reset_index()
)

sensitivity_aggregate.to_csv(
    TABLE_DIR /
    "mc_dropout_threshold_sensitivity_mean_sd.csv",
    index=False,
)

display(
    sensitivity_aggregate
)


,seed,requested_val_false_referral_rate,threshold_U_N,actual_val_false_referral_rate,familiar_test_false_referral_rate,normal_test_false_referral_rate,known_only_test_false_referral_rate,unknown_referral_sensitivity,post_referral_rare_to_normal_rate,post_referral_explicit_normal_false_reassurance
0,42,0.05,1.790373,0.05,0.050788,0.000000,0.066362,0.12500,0.21875,0.09375
1,42,0.10,1.301160,0.10,0.108581,0.000000,0.141876,0.18750,0.21875,0.09375
2,42,0.15,1.027068,0.15,0.148862,0.000000,0.194508,0.43750,0.15625,0.09375
3,123,0.05,1.786734,0.05,0.042032,0.000000,0.054920,0.15625,0.18750,0.15625
4,123,0.10,1.315754,0.10,0.064799,0.000000,0.084668,0.21875,0.12500,0.12500
5,123,0.15,1.042731,0.15,0.082312,0.000000,0.107551,0.28125,0.12500,0.12500
6,2026,0.05,1.689866,0.05,0.050788,0.000000,0.066362,0.34375,0.21875,0.18750
7,2026,0.10,1.335869,0.10,0.078809,0.000000,0.102975,0.37500,0.21875,0.18750
8,2026,0.15,1.017458,0.15,0.126095,0.007463,0.162471,0.43750,0.18750,0.18750


,requested_val_false_referral_rate,familiar_false_referral_mean,familiar_false_referral_sd,normal_false_referral_mean,known_false_referral_mean,unknown_referral_sensitivity_mean,unknown_referral_sensitivity_sd,rare_to_normal_mean,strict_false_reassurance_mean
0,0.05,0.047869,0.005056,0.000000,0.062548,0.208333,0.118311,0.208333,0.145833
1,0.10,0.084063,0.022359,0.000000,0.109840,0.260417,0.100455,0.187500,0.135417
2,0.15,0.119089,0.033823,0.002488,0.154844,0.385417,0.090211,0.156250,0.135417


# 16. Safety Improvement Summary


In [16]:
# ============================================================
# 16. Safety improvement
# ============================================================

baseline_rtn = float(
    seed_results_df[
        "closedset_rare_to_normal_rate"
    ].mean()
)

post_rtn = float(
    seed_results_df[
        "post_referral_rare_to_normal_rate"
    ].mean()
)

absolute_reduction = (
    baseline_rtn
    - post_rtn
)

relative_reduction = (
    absolute_reduction
    / baseline_rtn
    if baseline_rtn > 0
    else np.nan
)

summary = {
    "closedset_rare_to_normal_rate":
        baseline_rtn,

    "mc_dropout_post_referral_rare_to_normal_rate":
        post_rtn,

    "absolute_reduction":
        absolute_reduction,

    "relative_reduction":
        relative_reduction,

    "mean_unknown_referral_sensitivity":
        float(
            seed_results_df[
                "unknown_referral_sensitivity"
            ].mean()
        ),

    "mean_familiar_false_referral_rate":
        float(
            seed_results_df[
                "familiar_test_false_referral_rate"
            ].mean()
        ),
}

print(
    json.dumps(
        summary,
        indent=2,
    )
)

with open(
    TABLE_DIR /
    "mc_dropout_primary_safety_summary.json",
    "w",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )


{
  "closedset_rare_to_normal_rate": 0.2708333333333333,
  "mc_dropout_post_referral_rare_to_normal_rate": 0.1875,
  "absolute_reduction": 0.08333333333333331,
  "relative_reduction": 0.30769230769230765,
  "mean_unknown_referral_sensitivity": 0.2604166666666667,
  "mean_familiar_false_referral_rate": 0.0840630472854641
}


# 17. Save Method Metadata


In [17]:
# ============================================================
# 17. Metadata
# ============================================================

metadata = {
    "project":
        "SafeFundus",

    "classifier":
        "Swin-Tiny + ASL + explicit NORMAL output",

    "uncertainty_method":
        "Prediction-head MC Dropout",

    "mc_passes":
        CONFIG[
            "mc_passes"
        ],

    "dropout_probability":
        CONFIG[
            "dropout_probability"
        ],

    "primary_uncertainty":
        "mean predictive probability variance across 15 outputs",

    "normalization":
        "z-score using familiar validation uncertainty only",

    "primary_referral_threshold":
        "90th percentile of normalized familiar validation uncertainty",

    "primary_validation_false_referral_target":
        CONFIG[
            "primary_val_false_referral_rate"
        ],

    "classifier_decision_for_primary_safety_metric":
        "deterministic 06C probabilities with validation thresholds",

    "unknown_test_used_for_tuning":
        False,

    "next_step":
        "Fuse Normal-Aware JointEnergy with normalized MC Dropout uncertainty",
}

with open(
    OUTPUT_DIR /
    "mc_dropout_method_metadata.json",
    "w",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

print(
    json.dumps(
        metadata,
        indent=2,
    )
)


{
  "project": "SafeFundus",
  "classifier": "Swin-Tiny + ASL + explicit NORMAL output",
  "uncertainty_method": "Prediction-head MC Dropout",
  "mc_passes": 20,
  "dropout_probability": 0.3,
  "primary_uncertainty": "mean predictive probability variance across 15 outputs",
  "normalization": "z-score using familiar validation uncertainty only",
  "primary_referral_threshold": "90th percentile of normalized familiar validation uncertainty",
  "primary_validation_false_referral_target": 0.1,
  "classifier_decision_for_primary_safety_metric": "deterministic 06C probabilities with validation thresholds",
  "unknown_test_used_for_tuning": false,
  "next_step": "Fuse Normal-Aware JointEnergy with normalized MC Dropout uncertainty"
}


# 18. Create One Complete Output ZIP

The notebook automatically creates:

**`08_MC_Dropout_Uncertainty.zip`**

The ZIP contains:

- all per-image MC mean probabilities;
- all per-class predictive variances;
- primary all-15 uncertainty;
- disease-only and NORMAL uncertainty diagnostics;
- predictive entropy;
- validation normalization statistics;
- frozen validation referral thresholds;
- familiar/normal/known false-referral rates;
- unknown referral sensitivity;
- rare-to-normal safety results;
- mixed/intermediate exploratory referral rates;
- AUROC / AUPR / FPR@95TPR;
- 5% / 10% / 15% threshold sensitivity;
- multi-seed mean ± SD;
- score-distribution figures.

The neural-network checkpoints are not duplicated because they remain in Notebook 06C.


In [18]:
# ============================================================
# 18. ZIP output
# ============================================================

ZIP_PATH = (
    WORK_ROOT /
    f"{NOTEBOOK_STEM}.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(
        ZIP_PATH.with_suffix("")
    ),
    format="zip",
    root_dir=OUTPUT_DIR,
)

size_mb = (
    ZIP_PATH.stat().st_size
    / (1024 ** 2)
)

print("=" * 80)
print("COMPLETE OUTPUT ZIP CREATED")
print("=" * 80)

print(ZIP_PATH)

print(
    f"ZIP size: "
    f"{size_mb:.2f} MB"
)


COMPLETE OUTPUT ZIP CREATED
/kaggle/working/08_MC_Dropout_Uncertainty.zip
ZIP size: 0.76 MB


# Next Step

Upload:

**`08_MC_Dropout_Uncertainty.zip`**

I will compare:

1. Normal-Aware JointEnergy alone;
2. MC Dropout alone;
3. their error overlap and complementarity.

If they provide complementary signals, the next notebook will implement the final SafeFundus fusion:

\[
R(x)
=
0.5J_N(x)
+
0.5U_N(x)
\]

with all normalization and the final referral threshold fit on validation only.

That fusion notebook will also include the required ablation:

- classifier only;
- JointEnergy only;
- MC Dropout only;
- JointEnergy + MC Dropout fusion.
